# Step 4 — Which genes drive the step 3 senescence difference? (EXPLORATORY)

**Project:** pd-da-aging · **Status:** exploratory (post hoc). This analysis was decided on 26 Nov 2026, *after* seeing the step 3 results, and is logged as such in docs/ANALYSIS_PLAN.md. It explains the H1 result; it does not test a new pre-registered hypothesis.

Run the cells **one at a time, top to bottom**. Stop at each 🛑 and send Claude a screenshot.

**Needs:** the `pd-da-aging_step3` folder in *My Drive* (made in step 3). **Results go to:** *My Drive › pd-da-aging_step4*. Runtime: CPU is fine.

## Cell 1 — Connect Google Drive and install tools
Click **Allow** when Google asks. If Colab warns the notebook was not written by Google, click **Run anyway**.

In [ ]:
# Connect Google Drive and install tools (1-2 minutes)
from google.colab import drive
drive.mount("/content/drive")
!pip install -q scanpy openpyxl statsmodels


## Cell 2 — Load the step 3 neurons
Uses `pd-da-aging_step3/GSE178265_DA_neurons_step3.h5ad` and the gene lists saved in step 3. Nothing is downloaded.

In [ ]:
# Set up, record versions and load the step 3 neurons (1-2 minutes)
import os, sys, io, json, glob, datetime, platform, urllib.request, warnings
import numpy as np, pandas as pd, scipy, scanpy as sc, anndata as ad, statsmodels
from scipy import stats, sparse
from statsmodels.stats.multitest import multipletests
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=FutureWarning)

DRIVE = "/content/drive/MyDrive"
IN = f"{DRIVE}/pd-da-aging_step3"
OUT = f"{DRIVE}/pd-da-aging_step4"
for sub in ["", "/figures", "/tables"]:
    os.makedirs(OUT + sub, exist_ok=True)
TODAY = datetime.date.today().isoformat()
versions = {"date": TODAY, "python": sys.version.split()[0], "platform": platform.platform()}
for m in [np, pd, scipy, sc, ad, statsmodels, matplotlib]:
    versions[m.__name__] = getattr(m, "__version__", "unknown")
with open(f"{OUT}/software_versions.txt", "w") as fh:
    fh.write("\n".join(f"{k}: {v}" for k, v in versions.items()) + "\n")

da = ad.read_h5ad(f"{IN}/GSE178265_DA_neurons_step3.h5ad")
gs_file = sorted(glob.glob(f"{IN}/gene_sets/gene_sets_*.json"))[-1]
GS = json.load(open(gs_file))
gene_sets, sources = GS["gene_sets"], GS["sources"]
for c in ["donor", "status", "group", "subtype"]:
    da.obs[c] = da.obs[c].astype(str)
counts = da.X.tocsr() if sparse.issparse(da.X) else sparse.csr_matrix(da.X)
lib = np.asarray(counts.sum(axis=1)).ravel()
lognorm = sparse.diags(1e4 / np.maximum(lib, 1)) @ counts
lognorm.data = np.log1p(lognorm.data)
print(f"{da.n_obs:,} neurons x {da.n_vars:,} genes; gene lists from {os.path.basename(gs_file)}")
print("Results will be saved in My Drive > pd-da-aging_step4")


## Cell 3 — Average every gene per donor and neuron group
Same 7 control donors and the same ≥ 20-neuron rule as H1.

In [ ]:
# Average every gene per donor and neuron group, in the same control donors as H1 (seconds)
MIN_CELLS = 20
d = da.obs[(da.obs["status"] == "Ctrl") & da.obs["group"].isin(["vulnerable", "resilient"])]
n = d.groupby(["donor", "group"]).size().unstack(fill_value=0)
donors = list(n.index[(n["vulnerable"] >= MIN_CELLS) & (n["resilient"] >= MIN_CELLS)])
print(f"Control donors used: {len(donors)} ({', '.join(donors)}); same rule as H1 (>= {MIN_CELLS} neurons per group)")

def group_stats(donor, group):
    idx = np.flatnonzero((da.obs["donor"] == donor).to_numpy() & (da.obs["group"] == group).to_numpy())
    ln, ct = lognorm[idx], counts[idx]
    mean_ln = np.asarray(ln.mean(axis=0)).ravel()                   # mean log-normalised expression
    detected = np.asarray((ct > 0).mean(axis=0)).ravel()             # fraction of neurons with >=1 read
    pb = np.asarray(ct.sum(axis=0)).ravel()
    log2cpm = np.log2(pb / pb.sum() * 1e6 + 1)                       # pseudobulk
    return mean_ln, detected, log2cpm

G = da.n_vars
M = {k: np.zeros((len(donors), G)) for k in ["ln_v", "ln_r", "det_v", "det_r", "pb_v", "pb_r"]}
for i, dn in enumerate(donors):
    M["ln_v"][i], M["det_v"][i], M["pb_v"][i] = group_stats(dn, "vulnerable")
    M["ln_r"][i], M["det_r"][i], M["pb_r"][i] = group_stats(dn, "resilient")
D_ln = M["ln_v"] - M["ln_r"]          # donors x genes: vulnerable minus resilient, log-normalised
D_pb = M["pb_v"] - M["pb_r"]          # donors x genes: log2 fold change, pseudobulk
print("Per-gene, per-donor averages ready.")


## Cell 4 — Gene-by-gene breakdown
Which SenMayo and CellAge genes make the vulnerable neurons score lower?
### 🛑 Send Claude a screenshot of this output.

In [ ]:
# Gene-by-gene breakdown of the SenMayo and CellAge differences (seconds)
genes = np.array(da.var_names)
gidx = {g: i for i, g in enumerate(genes)}

def gene_table(set_name, genes_used):
    rows = []
    k = len(donors)
    for g in genes_used:
        j = gidx[g]
        dl, dp = D_ln[:, j], D_pb[:, j]
        t = stats.ttest_1samp(dp, 0.0) if np.ptp(dp) > 0 else None
        se = dp.std(ddof=1) / np.sqrt(k)
        rows.append({"gene": g,
                     "detected_vulnerable_%": 100 * M["det_v"][:, j].mean(),
                     "detected_resilient_%": 100 * M["det_r"][:, j].mean(),
                     "mean_lognorm_vulnerable": M["ln_v"][:, j].mean(),
                     "mean_lognorm_resilient": M["ln_r"][:, j].mean(),
                     "diff_lognorm": dl.mean(),
                     "contribution_to_set_gap": dl.mean() / len(genes_used),
                     "log2FC_pseudobulk": dp.mean(),
                     "ci_low": dp.mean() - stats.t.ppf(0.975, k - 1) * se,
                     "ci_high": dp.mean() + stats.t.ppf(0.975, k - 1) * se,
                     "p": t.pvalue if t is not None else np.nan,
                     "donors_lower_in_vulnerable": int((dp < 0).sum()), "donors": k})
    tab = pd.DataFrame(rows).set_index("gene")
    ok = tab["p"].notna()
    tab.loc[ok, "q"] = multipletests(tab.loc[ok, "p"], method="fdr_bh")[1]
    tab = tab.sort_values("contribution_to_set_gap")
    tab.to_csv(f"{OUT}/tables/genes_{set_name}.csv")
    return tab

# Same genes as the step 3 scores: in the data, subtype marker genes removed
markers = set(gene_sets["kamath_marker_panel"])    # every subtype-defining gene is in this panel
used = {s: [g for g in gene_sets[s] if g in gidx and g not in markers] for s in ["SenMayo", "CellAge_induces"] if gene_sets.get(s)}
tabs = {s: gene_table(s, gs) for s, gs in used.items()}

pd.set_option("display.width", 250)
for s, tab in tabs.items():
    gap = tab["diff_lognorm"].mean()
    neg = tab["contribution_to_set_gap"].clip(upper=0)
    share = (neg.cumsum() / neg.sum())
    n80 = int((share < 0.8).sum()) + 1
    print(f"\n===== {s}: {len(tab)} genes =====")
    print(f"Average gene difference (vulnerable minus resilient, log-normalised): {gap:.4f}")
    print(f"Genes lower in vulnerable: {(tab['diff_lognorm'] < 0).sum()}; higher: {(tab['diff_lognorm'] > 0).sum()}")
    print(f"Genes lower in all {len(donors)} donors: {(tab['donors_lower_in_vulnerable'] == len(donors)).sum()}; "
          f"higher in all: {(tab['donors_lower_in_vulnerable'] == 0).sum()}")
    print(f"Pseudobulk genes with q < 0.05: lower {((tab['q'] < 0.05) & (tab['log2FC_pseudobulk'] < 0)).sum()}, "
          f"higher {((tab['q'] < 0.05) & (tab['log2FC_pseudobulk'] > 0)).sum()}")
    print(f"Genes detected in < 10% of neurons in both groups: "
          f"{((tab['detected_vulnerable_%'] < 10) & (tab['detected_resilient_%'] < 10)).sum()}")
    print(f"{n80} genes account for 80% of the downward pull")
    cols = ["detected_vulnerable_%", "detected_resilient_%", "diff_lognorm", "log2FC_pseudobulk", "ci_low", "ci_high",
            "p", "q", "donors_lower_in_vulnerable"]
    print("\nTop 15 genes pulling the score DOWN in vulnerable neurons:")
    print(tab[cols].head(15).round(3).to_string())
    print("\nTop 5 genes pulling it UP:")
    print(tab[cols].tail(5).iloc[::-1].round(3).to_string())


## Cell 5 — Is the gap special to senescence genes?
Compares the senescence gene sets with 1,000 random gene sets of matched expression level.
### 🛑 Send Claude a screenshot of the printed lines and both plots.

In [ ]:
# Is the gap special to senescence genes? Compare with 1,000 random gene sets of matched expression (1 minute)
expressed = (M["det_v"].mean(0) + M["det_r"].mean(0)) / 2 > 0.01      # detected in >1% of neurons
avg_expr = (M["ln_v"].mean(0) + M["ln_r"].mean(0)) / 2
gene_gap = D_ln.mean(0)                                                  # donor-averaged difference per gene
pool = np.flatnonzero(expressed)
bins = pd.qcut(avg_expr[pool], 25, labels=False, duplicates="drop")
by_bin = {b: pool[bins == b] for b in np.unique(bins)}
bin_of = dict(zip(pool, bins))
rng = np.random.default_rng(0)
null_rows = []
print(f"Background: {len(pool):,} genes detected in > 1% of neurons, in {len(by_bin)} expression bins\n")
for s, gl in used.items():
    idx = np.array([gidx[g] for g in gl if expressed[gidx[g]]])
    obs_gap = gene_gap[idx].mean()
    null = np.array([gene_gap[[rng.choice(by_bin[bin_of[j]]) for j in idx]].mean() for _ in range(1000)])
    pct = (null <= obs_gap).mean()
    null_rows.append({"set": s, "genes_expressed": len(idx), "observed_gap": obs_gap, "null_mean": null.mean(),
                      "null_2.5%": np.percentile(null, 2.5), "null_97.5%": np.percentile(null, 97.5),
                      "share_of_random_sets_as_low_or_lower": pct})
    print(f"{s}: {len(idx)} expressed genes; observed gap {obs_gap:.4f}; random matched sets: mean {null.mean():.4f} "
          f"(95% range {np.percentile(null, 2.5):.4f} to {np.percentile(null, 97.5):.4f}); "
          f"{pct:.1%} of random sets are as low or lower")
    plt.figure(figsize=(5, 3))
    plt.hist(null, bins=40, color="lightgrey", edgecolor="grey")
    plt.axvline(obs_gap, color="crimson", lw=2, label=f"{s} (observed)")
    plt.axvline(0, color="black", lw=0.8)
    plt.xlabel("Vulnerable minus resilient (mean over genes)"); plt.ylabel("Random gene sets")
    plt.title(f"{s} vs 1,000 expression-matched random sets", fontsize=9); plt.legend(fontsize=8)
    plt.savefig(f"{OUT}/figures/fig_null_{s}.png", dpi=200, bbox_inches="tight"); plt.show()
pd.DataFrame(null_rows).to_csv(f"{OUT}/tables/random_set_comparison.csv", index=False)


## Cell 6 — SenMayo genes by category
Uses the categories in the original SenMayo table (re-downloaded from the same source as step 3).
### 🛑 Send Claude a screenshot.

In [ ]:
# SenMayo genes by their category in the original paper (seconds)
cat = None
try:
    req = urllib.request.Request(sources["SenMayo"], headers={"User-Agent": "Mozilla/5.0"})
    sheets = pd.read_excel(io.BytesIO(urllib.request.urlopen(req, timeout=120).read()), sheet_name=None)
    for name, df in sheets.items():
        ccol = [c for c in df.columns if "class" in str(c).lower() or "categ" in str(c).lower()]
        gcol = [c for c in df.columns if df[c].astype(str).str.strip().isin(used["SenMayo"]).sum() > 50]
        if gcol and ccol:
            cat = df[[gcol[0], ccol[0]]].dropna().astype(str).apply(lambda x: x.str.strip())
            cat.columns = ["gene", "category"]
            cat = cat.drop_duplicates("gene").set_index("gene")["category"]
            print(f"Categories read from sheet '{name}', column '{ccol[0]}'")
            break
except Exception as e:
    print(f"Could not read SenMayo categories: {e}")
if cat is not None:
    t = tabs["SenMayo"].join(cat, how="left")
    t["category"] = t["category"].fillna("not listed")
    summ = t.groupby("category").agg(genes=("diff_lognorm", "size"),
                                     mean_detected_pct=("detected_resilient_%", "mean"),
                                     mean_gap=("diff_lognorm", "mean"),
                                     total_contribution=("contribution_to_set_gap", "sum"),
                                     genes_lower=("diff_lognorm", lambda x: int((x < 0).sum())))
    summ = summ.sort_values("total_contribution")
    print(summ.round(4).to_string())
    summ.to_csv(f"{OUT}/tables/SenMayo_by_category.csv")
    t.to_csv(f"{OUT}/tables/genes_SenMayo.csv")
else:
    print("No category column found; skipped.")


## Cell 7 — Figures and file list
### 🛑 Send Claude a screenshot of the two figures and the file list.

In [ ]:
# Figures: the genes behind the gap (seconds)
for s, tab in tabs.items():
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    top = pd.concat([tab.head(15), tab.tail(5)])
    colors = ["#3b6fb6" if v < 0 else "#c44e52" for v in top["contribution_to_set_gap"]]
    ax[0].barh(top.index[::-1], top["contribution_to_set_gap"][::-1], color=colors[::-1])
    ax[0].axvline(0, color="black", lw=0.8)
    ax[0].set_xlabel("Contribution to the set gap (vulnerable minus resilient)")
    ax[0].set_title(f"{s}: 15 genes pulling down, 5 pulling up", fontsize=9)
    x, y = tab["log2FC_pseudobulk"], -np.log10(tab["p"].clip(lower=1e-12))
    sig = tab["q"] < 0.05
    ax[1].scatter(x[~sig], y[~sig], s=12, color="lightgrey")
    ax[1].scatter(x[sig], y[sig], s=16, color="#3b6fb6")
    for g in tab.index[sig][:12]:
        ax[1].annotate(g, (x[g], y[g]), fontsize=7)
    ax[1].axvline(0, color="black", lw=0.8)
    ax[1].set_xlabel("log2 fold change, pseudobulk (vulnerable / resilient)"); ax[1].set_ylabel("-log10 p (paired, donors)")
    ax[1].set_title(f"{s}: each gene, {len(donors)} control donors (blue: q < 0.05)", fontsize=9)
    plt.tight_layout(); plt.savefig(f"{OUT}/figures/fig_genes_{s}.png", dpi=200, bbox_inches="tight"); plt.show()

for root, _, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f); print(f"{os.path.getsize(p) / 1e6:8.2f} MB  {p.replace(DRIVE + '/', '')}")
